### ZARR based implementation of preprocessing step for a window (SNN)

In [12]:
from destruction_utilities import *

In [13]:
CITY = 'test'
PRE_IMG_INDEX = 0
TILE_SIZE = (128,128)

samples = search_data(f'{CITY}_samples.tif$')
window  = center_window(samples, (20, 20))
samples = read_raster(f'../data/{CITY}/others/{CITY}_samples.tif', window=window)
images  = search_data(pattern(city=CITY, type='image'))
labels  = search_data(pattern(city=CITY, type='label'))

delete_zarr_if_exists(CITY, 'labels_siamese_train')
delete_zarr_if_exists(CITY, 'labels_siamese_test')
delete_zarr_if_exists(CITY, 'labels_siamese_valid')
delete_zarr_if_exists(CITY, 'images_siamese_train_tt')
delete_zarr_if_exists(CITY, 'images_siamese_test_tt')
delete_zarr_if_exists(CITY, 'images_siamese_valid_tt')
delete_zarr_if_exists(CITY, 'images_siamese_train_t0')
delete_zarr_if_exists(CITY, 'images_siamese_test_t0')
delete_zarr_if_exists(CITY, 'images_siamese_valid_t0')

window = center_window(images[0], (20*TILE_SIZE[0], 20*TILE_SIZE[1]))
pre_image = read_raster(images[PRE_IMG_INDEX], window=window, dtype='uint8')
pre_image = tile_sequences(np.array([pre_image]), TILE_SIZE)

for i in range(len(images)):
    if i != PRE_IMG_INDEX:
        window = center_window(labels[0], (20*1, 20*1))
        label = np.array(read_raster(labels[i], window=window, dtype='int8'))
        label = label.flatten()
        exclude = np.where(label==-1.0)
        label = np.delete(label, exclude)
        samples_valid = np.delete(samples.flatten(), exclude)
        _, label_train, label_test, label_valid = sample_split(label, samples_valid )
        save_zarr(np.equal(label_train, 3), CITY, 'labels_siamese_train')
        save_zarr(np.equal(label_test, 3), CITY, 'labels_siamese_test')
        save_zarr(np.equal(label_valid, 3), CITY, 'labels_siamese_valid')


        window = center_window(images[0], (20*TILE_SIZE[0], 20*TILE_SIZE[1]))
        image = read_raster(images[i], window=window, dtype='uint8')
        image = tile_sequences(np.array([image]), TILE_SIZE)
        image = np.delete(image, exclude, 0)
        _, image_train, image_test, image_valid = sample_split(image, samples_valid)
        save_zarr(flatten_image(image_train), CITY, 'images_siamese_train_tt')
        save_zarr(flatten_image(image_test), CITY, 'images_siamese_test_tt')
        save_zarr(flatten_image(image_valid), CITY, 'images_siamese_valid_tt')

        pre_image_v = np.delete(pre_image, exclude, 0)
        _, pre_image_train, pre_image_test, pre_image_valid = sample_split(pre_image_v, samples_valid)
        save_zarr(flatten_image(pre_image_train), CITY, 'images_siamese_train_t0')
        save_zarr(flatten_image(pre_image_test), CITY, 'images_siamese_test_t0')
        save_zarr(flatten_image(pre_image_valid), CITY, 'images_siamese_valid_t0')

# Generate a balanced (upsampled) dataset and shuffle it..
print('--- Generate a balanced (upsampled) dataset and shuffle it..')
delete_zarr_if_exists(CITY, 'labels_siamese_train_balanced')
delete_zarr_if_exists(CITY, 'images_siamese_train_t0_balanced')
delete_zarr_if_exists(CITY, 'images_siamese_train_t1_balanced')
balance_snn(CITY)

--- Generate a balanced (upsampled) dataset and shuffle it..


### ZARR based implementation of preprocessing step for a window (CNN)

In [9]:
CITY = 'test'
TILE_SIZE = (128,128)
MODE = 'all'

# CNN Settings
ZERO_DAMAGE_BEFORE_YEAR = 2012



In [10]:
import numpy as np
import geopandas
import rasterio
import re
import zarr
import os
from rasterio import features
from destruction_utilities import *
import time
import shutil


In [11]:
# Extract report dates
dates = search_data(pattern(city=CITY, type='image'))
dates = extract(dates, '\d{4}_\d{2}_\d{2}')
dates= list(map(lambda x: x.replace("_", "-"), dates))

samples = search_data(f'{CITY}_samples.tif$')
window  = center_window(samples, (50, 50))
samples = read_raster(f'../data/{CITY}/others/{CITY}_samples.tif', window=window)

delete_zarr_if_exists(CITY, 'labels_conv_train')
delete_zarr_if_exists(CITY, 'labels_conv_valid')
delete_zarr_if_exists(CITY, 'labels_conv_test')
delete_zarr_if_exists(CITY, 'images_conv_train')
delete_zarr_if_exists(CITY, 'images_conv_valid')
delete_zarr_if_exists(CITY, 'images_conv_test')

images  = search_data(pattern(city=CITY, type='image'))
labels  = search_data(pattern(city=CITY, type='label'))

for i in range(len(images)):
    window = center_window(labels[0], (50*1, 50*1))
    label = np.array(read_raster(labels[i], window=window))
    w, h, b = label.shape
    label = label.reshape(1, w, h, b)
    label = tile_sequences(label, tile_size=(1,1))
    exclude = np.where(label.flatten() == -1)
    label = np.delete(label, exclude, 0)
    label[label!=3.0] = 0.0
    label[label==3.0] = 1.0
    _, train, validate, test = sample_split(label, np.delete(samples.flatten(), exclude))
    train_shuffle = np.arange(len(train))
    validate_shuffle = np.arange(len(validate))
    test_shuffle = np.arange(len(test))
    np.random.shuffle(train_shuffle)
    np.random.shuffle(validate_shuffle)
    np.random.shuffle(test_shuffle)
    save_zarr(train[train_shuffle].reshape(np.take(train.shape, [0,2,3,4])), CITY, 'labels_conv_train')
    save_zarr(validate[validate_shuffle].reshape(np.take(validate.shape, [0,2,3,4])), CITY, 'labels_conv_valid')
    save_zarr(test[test_shuffle].reshape(np.take(test.shape, [0,2,3,4])), CITY, 'labels_conv_test')
    del _, train, validate, test, label

    window = center_window(images[0], (50*TILE_SIZE[0], 50*TILE_SIZE[1]))
    image = np.array(read_raster(images[i], window=window))
    w,h,b = image.shape
    image = image.reshape(1,w,h,b)
    image = tile_sequences(image,  tile_size=TILE_SIZE)
    image = np.delete(image, exclude, 0)
    _, train, validate, test = sample_split(image, np.delete(samples.flatten(), exclude))
    save_zarr(train[train_shuffle].reshape(np.take(train.shape, [0,2,3,4])), CITY, 'images_conv_train')
    save_zarr(validate[validate_shuffle].reshape(np.take(validate.shape, [0,2,3,4])), CITY,'images_conv_valid')
    save_zarr(test[test_shuffle].reshape(np.take(test.shape, [0,2,3,4])), CITY,'images_conv_test') 
    del _, train, validate, test, image, exclude
    print(f'------ {dates[i]}')
    gc.collect(generation=2)
del samples, images, labels

# Generate a balanced (upsampled) dataset and shuffle it..
print('--- Generate a balanced (upsampled) dataset and shuffle it..')
delete_zarr_if_exists(CITY, 'labels_conv_train_balanced')
delete_zarr_if_exists(CITY, 'images_conv_train_balanced')
delete_zarr_if_exists(CITY, 'labels_conv_train_balanced_shuffled')
delete_zarr_if_exists(CITY, 'images_conv_train_balanced_shuffled')
balance(CITY)
shuffle(CITY, TILE_SIZE, (1000,7500))

IndexError: list index out of range

### Clement's implementation

In [ ]:
# Modules
import numpy as np
import destruction_models as models

from destruction_utilities import *
from keras import callbacks, metrics

#%% LOADS DATA

# Reads images as sequences (20x20 tile subset)
tile_size = (128, 128)
images  = search_data(pattern('test', 'image'))
window  = center_window(images[0], (20*tile_size[0], 20*tile_size[1]))
images  = np.array([read_raster(image, window=window, dtype='uint8') for image in images])
images  = tile_sequences(images, tile_size)
del window

# Reads labels as sequences (20x20 tile subset)
tile_size = (1, 1)
labels = search_data(pattern('test', 'label'))
window = center_window(labels[0], (20*tile_size[0], 20*tile_size[1]))
labels = np.array([read_raster(label, window=window, dtype='uint8') for label in labels])
labels = np.equal(labels, 3)
labels = tile_sequences(labels, tile_size)
del window

# Reads samples (20x20 tile subset)
samples = search_data('test_samples.tif$')
window  = center_window(samples, (20, 20))
samples = read_raster(samples, window=window, dtype='int8')
samples = samples.flatten()
del window

#%% RESHAPES DATA FOR SIAMESE MODEL

def reshape_siamese(images):
    n, t, h, w, d = images.shape
    images_t0 = np.tile(np.take(images, [0], 1), (1, t-1, 1, 1, 1)).reshape(n * (t-1), h, w, d)
    images_tt = np.delete(images, 0, 1).reshape(n * (t-1), h, w, d)
    return images_t0, images_tt

# Split samples
_, images_train, images_test, images_valid = sample_split(images, samples)
_, labels_train, labels_test, labels_valid = sample_split(labels, samples)
del _

images_train_t0, images_train_tt = reshape_siamese(images_train)
images_valid_t0, images_valid_tt = reshape_siamese(images_valid)
labels_train = np.delete(labels_train, 0, 1).flatten()
labels_valid = np.delete(labels_valid, 0, 1).flatten()

#%% ESTIMATES SIAMESE MODEL

# Model structure
model = models.siamese_convolutional_network(shape=(128, 128, 3), args_encode=dict(filters=8, dropout=0), args_dense=dict(units=16, dropout=0))
model.compile(optimizer='adam', loss='binary_focal_crossentropy', metrics=metrics.AUC(num_thresholds=200, curve='ROC'))
model.summary()

# Estimates parameters
training = model.fit(    
    {'images_t0':images_train_t0, 'images_tt':images_train_tt}, 
    y=labels_train,
    epochs=100,
    verbose=1,
    callbacks=callbacks.EarlyStopping(monitor='val_auc', patience=5, restore_best_weights=True)
)
